# Agent 1 — Validation Agent

Runs all configured deterministic checks against registered tables
for a given business date. Reads configuration from metadata tables,
compares observed values against baselines, scores results against
thresholds, and writes outcomes to validation result tables.

**Parameters:**
  business_date : Date to validate in YYYY-MM-DD format
  table_filter  : Optional. Fully qualified table name to validate
                  one specific table. Leave blank to validate all.

**Reads from:**
  uc1_table_registry_b
  uc1_column_registry_b
  uc1_column_validation_registry_b
  uc1_table_validation_registry_b
  uc1_baseline_metrics_b

**Writes to:**
  uc1_validation_run_b
  uc1_validation_results_b

## Parameters

When run manually:
  Set business_date to the date you want to validate
  Leave blank to default to yesterday

When run via Databricks Workflow:
  business_date is passed by the workflow trigger
  Blank means use yesterday (standard production behaviour)

In [0]:
dbutils.widgets.text(
    "business_date",
    "",
    "Business Date (YYYY-MM-DD, leave blank for yesterday)"
)

dbutils.widgets.text(
    "table_filter",
    "",
    "Table Filter (leave blank for all active tables)"
)

from datetime import date, datetime, timedelta

BUSINESS_DATE_STR = dbutils.widgets.get("business_date").strip()
TABLE_FILTER      = dbutils.widgets.get("table_filter").strip()

# ── Resolve business date ──────────────────────────────────────────────────

if not BUSINESS_DATE_STR:
    BUSINESS_DATE = date.today() - timedelta(days=1)
    date_source   = "default (yesterday)"
else:
    try:
        BUSINESS_DATE = date.fromisoformat(BUSINESS_DATE_STR)
        date_source   = "provided parameter"
    except ValueError:
        print(
            f"WARNING: Invalid date format '{BUSINESS_DATE_STR}'. "
            f"Expected YYYY-MM-DD. Defaulting to yesterday."
        )
        BUSINESS_DATE = date.today() - timedelta(days=1)
        date_source   = "fallback (invalid format provided)"

RUN_START_TIME = datetime.now()

print("=" * 60)
print("AGENT 1 — VALIDATION AGENT")
print("=" * 60)
print(f"Business date  : {BUSINESS_DATE}  ({date_source})")
print(f"Table filter   : {TABLE_FILTER or 'All active tables'}")
print(f"Run IDs        : Generated per table inside the loop")
print(f"Started at     : {RUN_START_TIME}")
print("=" * 60)

AGENT 1 — VALIDATION AGENT
Business date  : 2026-10-14  (provided parameter)
Table filter   : All active tables
Run IDs        : Generated per table inside the loop
Started at     : 2026-11-11 16:31:22.283194


In [0]:
import uuid
import json
from datetime import date, datetime, timedelta
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DateType, TimestampType,
    DoubleType, IntegerType, BooleanType, LongType
)
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# ── Catalog and Schema ─────────────────────────────────────────────────────

dbutils.widgets.text("catalog", "learning_dev", "Target Catalog")
dbutils.widgets.text("schema", "s819534", "Target Schema")
CATALOG = dbutils.widgets.get("catalog")
SCHEMA  = dbutils.widgets.get("schema")

# ── Metadata Table References ──────────────────────────────────────────────

TABLE_REGISTRY             = f"{CATALOG}.{SCHEMA}.uc1_table_registry_b"
COLUMN_REGISTRY            = f"{CATALOG}.{SCHEMA}.uc1_column_registry_b"
COLUMN_VALIDATION_REGISTRY = f"{CATALOG}.{SCHEMA}.uc1_column_validation_registry_b"
TABLE_VALIDATION_REGISTRY  = f"{CATALOG}.{SCHEMA}.uc1_table_validation_registry_b"
BASELINE_METRICS           = f"{CATALOG}.{SCHEMA}.uc1_baseline_metrics_b"
VALIDATION_RUN             = f"{CATALOG}.{SCHEMA}.uc1_validation_run_b"
VALIDATION_RESULTS         = f"{CATALOG}.{SCHEMA}.uc1_validation_results_b"

# ── Severity Levels ────────────────────────────────────────────────────────

STATUS_PASS     = "PASS"
STATUS_INFO     = "INFO"
STATUS_WARNING  = "WARNING"
STATUS_CRITICAL = "CRITICAL"
STATUS_ERROR    = "ERROR"

# Severity rank for comparison (higher = worse)
SEVERITY_RANK = {
    STATUS_PASS    : 0,
    STATUS_INFO    : 1,
    STATUS_WARNING : 2,
    STATUS_CRITICAL: 3,
    STATUS_ERROR   : 4
}

# ── AI Confidence Threshold ────────────────────────────────────────────────

# AI is invoked when:
#   1. At least 2 checks have failed (WARNING or CRITICAL)
#   2. At least 1 check is CRITICAL
AI_MIN_FAILED_CHECKS  = 2
AI_MIN_CRITICAL_COUNT = 1

# ── Validation Run ID ──────────────────────────────────────────────────────

VALIDATION_RUN_ID = str(uuid.uuid4())
RUN_START_TIME    = datetime.now()

print("=" * 60)
print("AGENT 1 — VALIDATION AGENT")
print("=" * 60)
print(f"Business date     : {BUSINESS_DATE}")
print(f"Table filter      : {TABLE_FILTER or 'All active tables'}")
print(f"Validation run ID : {VALIDATION_RUN_ID}")
print(f"Started at        : {RUN_START_TIME}")
print("=" * 60)

AGENT 1 — VALIDATION AGENT
Business date     : 2026-10-14
Table filter      : All active tables
Validation run ID : 0b22e45c-65b8-4afc-80a0-baae6fb8410b
Started at        : 2026-11-11 16:31:22.433859


## Output Table Schemas

In [0]:
VALIDATION_RUN_SCHEMA = StructType([
    StructField("validation_run_id",           StringType(),    False),
    StructField("fully_qualified_name",        StringType(),    False),
    StructField("business_date",               DateType(),      False),
    StructField("ingestion_timestamp",         TimestampType(), True),
    StructField("triggered_by",                StringType(),    True),
    StructField("pipeline_run_id",             StringType(),    True),
    StructField("validation_start_timestamp",  TimestampType(), False),
    StructField("validation_end_timestamp",    TimestampType(), True),
    StructField("validation_duration_seconds", DoubleType(),    True),
    StructField("overall_status",              StringType(),    False),
    StructField("total_checks_run",            IntegerType(),   False),
    StructField("total_checks_passed",         IntegerType(),   False),
    StructField("total_checks_warned",         IntegerType(),   False),
    StructField("total_checks_critical",       IntegerType(),   False),
    StructField("total_checks_errored",        IntegerType(),   False),
    StructField("ai_enrichment_status",        StringType(),    False),
    StructField("ai_enrichment_timestamp",     TimestampType(), True),
    StructField("created_timestamp",           TimestampType(), False),
    StructField("created_by",                  StringType(),    False),
])

VALIDATION_RESULTS_SCHEMA = StructType([
    StructField("result_id",              StringType(),    False),
    StructField("validation_run_id",      StringType(),    False),
    StructField("fully_qualified_name",   StringType(),    False),
    StructField("business_date",          DateType(),      False),
    StructField("check_type",             StringType(),    False),
    StructField("check_level",            StringType(),    False),
    StructField("column_name",            StringType(),    True),
    StructField("check_status",           StringType(),    False),
    StructField("observed_value",         DoubleType(),    True),
    StructField("baseline_value",         DoubleType(),    True),
    StructField("deviation_percentage",   DoubleType(),    True),
    StructField("threshold_warning",      DoubleType(),    True),
    StructField("threshold_critical",     DoubleType(),    True),
    StructField("check_detail",           StringType(),    True),
    StructField("check_duration_seconds", DoubleType(),    True),
    StructField("ai_summary",             StringType(),    True),
    StructField("ai_recommended_action",  StringType(),    True),
    StructField("ai_failure_category",    StringType(),    True),
    StructField("ai_confidence_score",    DoubleType(),    True),
    StructField("ai_model_version",       StringType(),    True),
    StructField("ai_invocation_id",       StringType(),    True),
    StructField("created_timestamp",      TimestampType(), False),
    StructField("ai_enriched_timestamp",  TimestampType(), True),
])

print("Output schemas defined.")

Output schemas defined.


## Step 1: Load Configuration from Metadata Tables

In [0]:
def load_active_tables(table_filter: str = "") -> list:
    df = spark.table(TABLE_REGISTRY).filter(F.col("is_active") == True)
    if table_filter:
        df = df.filter(F.col("fully_qualified_name") == table_filter)
    # Convert Rows to dicts
    return [row.asDict() for row in df.collect()]


def load_active_columns(fqn: str) -> list:
    rows = (
        spark.table(COLUMN_REGISTRY)
             .filter(
                 (F.col("fully_qualified_name") == fqn) &
                 (F.col("is_active") == True)
             )
             .collect()
    )
    # Convert Rows to dicts
    return [row.asDict() for row in rows]


def load_column_validations(fqn: str) -> list:
    rows = (
        spark.table(COLUMN_VALIDATION_REGISTRY)
             .filter(
                 (F.col("fully_qualified_name") == fqn) &
                 (F.col("is_active") == True)
             )
             .collect()
    )
    # Convert Rows to dicts
    return [row.asDict() for row in rows]


def load_table_validations(fqn: str) -> list:
    rows = (
        spark.table(TABLE_VALIDATION_REGISTRY)
             .filter(
                 (F.col("fully_qualified_name") == fqn) &
                 (F.col("is_active") == True)
             )
             .collect()
    )
    # Convert Rows to dicts
    return [row.asDict() for row in rows]


def load_baseline(
    fqn         : str,
    check_type  : str,
    column_name : str = "__table__",
    metric_name : str = None
) -> dict:
    df = (
        spark.table(BASELINE_METRICS)
             .filter(
                 (F.col("fully_qualified_name") == fqn) &
                 (F.col("check_type") == check_type) &
                 (F.col("column_name") == column_name) &
                 (F.col("baseline_status").isin(["VALID", "INSUFFICIENT_DATA"]))
             )
             .orderBy(F.col("baseline_date").desc())
    )

    if metric_name:
        df = df.filter(F.col("metric_name") == metric_name)

    rows = df.limit(1).collect()
    if not rows:
        return None

    # Convert Row to dict
    row = rows[0].asDict()
    return {
        "mean_value"         : row.get("mean_value"),
        "stddev_value"       : row.get("stddev_value"),
        "min_value"          : row.get("min_value"),
        "max_value"          : row.get("max_value"),
        "p25_value"          : row.get("p25_value"),
        "p50_value"          : row.get("p50_value"),
        "p75_value"          : row.get("p75_value"),
        "p95_value"          : row.get("p95_value"),
        "sample_count"       : row.get("sample_count"),
        "is_sufficient_data" : row.get("is_sufficient_data"),
        "baseline_status"    : row.get("baseline_status")
    }


def parse_custom_parameters(custom_params_str: str) -> dict:
    if not custom_params_str:
        return {}
    try:
        return json.loads(custom_params_str)
    except (json.JSONDecodeError, TypeError):
        return {}


print("Configuration loaders ready.")

Configuration loaders ready.


In [0]:
import re
from pyspark.sql.types import ArrayType, StructType

# Registry paths follow the Unity Catalog schema view: the element of an
# array is called `items`, e.g. stores.selected.items.id.
# Legacy paths with [] (stores.selected[].id) are still accepted.
ARRAY_ELEMENT = "items"


def normalize_nested_path(path: str) -> str:
    p = (path or "").replace("[]", f".{ARRAY_ELEMENT}")
    while ".." in p:
        p = p.replace("..", ".")
    return p.strip(".")


def is_nested_path(column_name: str) -> bool:
    return "." in (column_name or "") or "[]" in (column_name or "")


def resolve_nested_column(df, column_name: str):
    """
    Resolves registry column paths like touchpoint.id or stores.selected.items.id
    into a DataFrame plus a materialized alias column name.
    Arrays are exploded with explode_outer so nested array elements can be
    validated like primitive columns.
    """
    if not column_name:
        return df, None
    if column_name in df.columns:
        return df, column_name

    # Schema-aware walk: `items` explodes only when the current type is an
    # array, so a struct field literally named `items` stays a field.
    segs = normalize_nested_path(column_name).split(".")
    root = segs[0]
    if root not in df.columns:
        return df, None
    cur = F.col(f"`{root}`")
    dt  = df.schema[root].dataType
    explode_idx = 0
    for seg in segs[1:]:
        if isinstance(dt, ArrayType) and seg == ARRAY_ELEMENT:
            explode_idx += 1
            tmp = f"__ds_exploded_{explode_idx}"
            df  = df.withColumn(tmp, F.explode_outer(cur))
            cur = F.col(tmp)
            dt  = dt.elementType
        elif isinstance(dt, StructType) and seg in dt.fieldNames():
            cur = cur.getField(seg)
            dt  = dt[seg].dataType
        else:
            print(f"      Field '{seg}' not found while resolving {column_name}.")
            return df, None

    alias = "__ds_" + re.sub(r"[^0-9A-Za-z_]", "_", normalize_nested_path(column_name))
    df = df.withColumn(alias, cur)
    return df, alias


print("Nested column helpers ready.")

Nested column helpers ready.


## Step 2: Check Scoring Logic

In [0]:
def score_check(
    observed_value    : float,
    baseline_value    : float,
    warning_threshold : float,
    critical_threshold: float,
    threshold_unit    : str,
    higher_is_worse   : bool = True
) -> tuple:
    """
    Scores a check result against warning and critical thresholds.

    Args:
        observed_value     : The value measured in the current load
        baseline_value     : The expected value from baseline
        warning_threshold  : Warning threshold value
        critical_threshold : Critical threshold value
        threshold_unit     : PERCENTAGE or COUNT or BOOLEAN
        higher_is_worse    : If True, higher observed = worse.
                             If False, lower observed = worse (volume drop).

    Returns:
        (status, deviation_percentage)
    """
    if observed_value is None or baseline_value is None:
        return STATUS_INFO, None

    if threshold_unit == "BOOLEAN":
        status = STATUS_CRITICAL if observed_value > 0 else STATUS_PASS
        return status, observed_value

    if threshold_unit == "PERCENTAGE":
        if baseline_value == 0:
            deviation = observed_value * 100
        else:
            deviation = abs(
                (observed_value - baseline_value) / baseline_value
            ) * 100

        if deviation >= critical_threshold:
            return STATUS_CRITICAL, deviation
        elif deviation >= warning_threshold:
            return STATUS_WARNING, deviation
        else:
            return STATUS_PASS, deviation

    if threshold_unit == "COUNT":
        deviation = abs(observed_value - baseline_value)
        if deviation >= critical_threshold:
            return STATUS_CRITICAL, deviation
        elif deviation >= warning_threshold:
            return STATUS_WARNING, deviation
        else:
            return STATUS_PASS, deviation

    if threshold_unit == "MINUTES":
        if observed_value >= critical_threshold:
            return STATUS_CRITICAL, observed_value
        elif observed_value >= warning_threshold:
            return STATUS_WARNING, observed_value
        else:
            return STATUS_PASS, observed_value

    return STATUS_PASS, 0.0


def score_rate_check(
    observed_rate     : float,
    baseline_rate     : float,
    warning_threshold : float,
    critical_threshold: float
) -> tuple:
    """
    Scores a rate-based check (null rate, violation rate, etc.)
    directly against percentage thresholds.

    The threshold is applied to the observed_rate directly,
    not to the deviation from baseline.

    Returns: (status, deviation_percentage)
    """
    if observed_rate is None:
        return STATUS_INFO, None

    observed_pct = observed_rate * 100

    if observed_pct >= critical_threshold:
        return STATUS_CRITICAL, observed_pct
    elif observed_pct >= warning_threshold:
        return STATUS_WARNING, observed_pct
    else:
        return STATUS_PASS, observed_pct


def build_result(
    validation_run_id  : str,
    fqn                : str,
    business_date      : date,
    check_type         : str,
    check_level        : str,
    column_name        : str,
    check_status       : str,
    observed_value     : float,
    baseline_value     : float,
    deviation_pct      : float,
    warning_threshold  : float,
    critical_threshold : float,
    check_detail       : dict,
    duration_seconds   : float
) -> dict:
    """
    Builds a single validation result record.
    """
    return {
        "result_id"             : str(uuid.uuid4()),
        "validation_run_id"     : validation_run_id,
        "fully_qualified_name"  : fqn,
        "business_date"         : business_date,
        "check_type"            : check_type,
        "check_level"           : check_level,
        "column_name"           : column_name,
        "check_status"          : check_status,
        "observed_value"        : observed_value,
        "baseline_value"        : baseline_value,
        "deviation_percentage"  : deviation_pct,
        "threshold_warning"     : warning_threshold,
        "threshold_critical"    : critical_threshold,
        "check_detail"          : json.dumps(check_detail),
        "check_duration_seconds": duration_seconds,
        "ai_summary"            : None,
        "ai_recommended_action" : None,
        "ai_failure_category"   : None,
        "ai_confidence_score"   : None,
        "ai_model_version"      : None,
        "ai_invocation_id"      : None,
        "created_timestamp"     : datetime.now(),
        "ai_enriched_timestamp" : None
    }


print("Scoring helpers ready.")


Scoring helpers ready.


## Step 3: Deterministic Check Implementations

In [0]:
def check_volume_anomaly(
    fqn               : str,
    business_date     : date,
    biz_date_col      : str,
    validation_config : dict,
    baseline          : dict,
    run_id            : str
) -> dict:
    """
    Check: VOLUME_ANOMALY
    Compares today's row count against the historical baseline mean.
    Flags both drops and spikes based on configured direction.
    """
    t_start = datetime.now()
    print(f"    [VOLUME_ANOMALY] Running...")

    warning   = float(validation_config["warning_threshold"])
    critical  = float(validation_config["critical_threshold"])
    unit      = validation_config["threshold_unit"]
    params    = parse_custom_parameters(validation_config.get("custom_parameters"))
    direction = params.get("direction", "both")

    try:
        df = spark.table(fqn)

        if biz_date_col and biz_date_col in df.columns:
            current_count = df.filter(
                F.col(biz_date_col) == F.lit(str(business_date))
            ).count()
        else:
            current_count = df.count()

        baseline_mean = baseline["mean_value"] if baseline else None

        if baseline_mean and baseline_mean > 0:
            deviation_pct = (
                (current_count - baseline_mean) / baseline_mean
            ) * 100
        else:
            deviation_pct = 0.0

        abs_deviation = abs(deviation_pct)

        if abs_deviation >= critical:
            status = STATUS_CRITICAL
        elif abs_deviation >= warning:
            status = STATUS_WARNING
        else:
            status = STATUS_PASS

        duration = (datetime.now() - t_start).total_seconds()

        detail = {
            "current_row_count"   : current_count,
            "baseline_mean"       : baseline_mean,
            "baseline_stddev"     : baseline["stddev_value"] if baseline else None,
            "deviation_pct"       : round(deviation_pct, 4),
            "direction"           : "DROP" if deviation_pct < 0 else "SPIKE",
            "baseline_min"        : baseline["min_value"] if baseline else None,
            "baseline_max"        : baseline["max_value"] if baseline else None
        }

        print(f"      Current rows    : {current_count}")
        print(f"      Baseline mean   : {baseline_mean}")
        print(f"      Deviation       : {deviation_pct:.2f}%")
        print(f"      Status          : {status}")

        return build_result(
            validation_run_id  = run_id,
            fqn                = fqn,
            business_date      = business_date,
            check_type         = "VOLUME_ANOMALY",
            check_level        = "TABLE",
            column_name        = None,
            check_status       = status,
            observed_value     = float(current_count),
            baseline_value     = baseline_mean,
            deviation_pct      = round(abs_deviation, 4),
            warning_threshold  = warning,
            critical_threshold = critical,
            check_detail       = detail,
            duration_seconds   = duration
        )

    except Exception as e:
        print(f"      ERROR: {str(e)}")
        return build_result(
            validation_run_id  = run_id,
            fqn                = fqn,
            business_date      = business_date,
            check_type         = "VOLUME_ANOMALY",
            check_level        = "TABLE",
            column_name        = None,
            check_status       = STATUS_ERROR,
            observed_value     = None,
            baseline_value     = None,
            deviation_pct      = None,
            warning_threshold  = warning,
            critical_threshold = critical,
            check_detail       = {"error": str(e)},
            duration_seconds   = (datetime.now() - t_start).total_seconds()
        )


def check_schema_conformance(
    fqn               : str,
    business_date     : date,
    validation_config : dict,
    baseline          : dict,
    run_id            : str
) -> dict:
    """
    Check: SCHEMA_CONFORMANCE
    Compares current column count against baseline column count.
    Detects added or dropped columns.
    """
    t_start  = datetime.now()
    print(f"    [SCHEMA_CONFORMANCE] Running...")

    warning  = float(validation_config["warning_threshold"])
    critical = float(validation_config["critical_threshold"])

    try:
        current_columns  = spark.table(fqn).columns
        current_count    = len(current_columns)
        baseline_count   = int(baseline["mean_value"]) if baseline else current_count

        added_count   = max(0, current_count - baseline_count)
        dropped_count = max(0, baseline_count - current_count)
        change_count  = added_count + dropped_count

        if dropped_count >= critical:
            status = STATUS_CRITICAL
        elif change_count >= critical:
            status = STATUS_CRITICAL
        elif change_count > warning:
            status = STATUS_WARNING
        else:
            status = STATUS_PASS

        duration = (datetime.now() - t_start).total_seconds()

        detail = {
            "current_column_count"  : current_count,
            "baseline_column_count" : baseline_count,
            "columns_added"         : added_count,
            "columns_dropped"       : dropped_count,
            "current_columns"       : current_columns
        }

        print(f"      Current columns  : {current_count}")
        print(f"      Baseline columns : {baseline_count}")
        print(f"      Status           : {status}")

        return build_result(
            validation_run_id  = run_id,
            fqn                = fqn,
            business_date      = business_date,
            check_type         = "SCHEMA_CONFORMANCE",
            check_level        = "TABLE",
            column_name        = None,
            check_status       = status,
            observed_value     = float(current_count),
            baseline_value     = float(baseline_count),
            deviation_pct      = float(change_count),
            warning_threshold  = warning,
            critical_threshold = critical,
            check_detail       = detail,
            duration_seconds   = duration
        )

    except Exception as e:
        print(f"      ERROR: {str(e)}")
        return build_result(
            validation_run_id  = run_id,
            fqn                = fqn,
            business_date      = business_date,
            check_type         = "SCHEMA_CONFORMANCE",
            check_level        = "TABLE",
            column_name        = None,
            check_status       = STATUS_ERROR,
            observed_value     = None,
            baseline_value     = None,
            deviation_pct      = None,
            warning_threshold  = warning,
            critical_threshold = critical,
            check_detail       = {"error": str(e)},
            duration_seconds   = (datetime.now() - t_start).total_seconds()
        )


def check_null_spike(
    fqn               : str,
    business_date     : date,
    biz_date_col      : str,
    column_name       : str,
    validation_config : dict,
    baseline          : dict,
    run_id            : str
) -> dict:
    """
    Check: NULL_SPIKE

    Scoring logic:
      For columns with a natural null rate (e.g. customer_id at 35%):
        Compares how much the null rate has INCREASED above baseline.
        warning_threshold = how many percentage points above baseline
        triggers a warning.

      For columns expected to have zero nulls (e.g. transaction_id):
        If warning_threshold = 0, any null triggers WARNING.
        If warning_threshold > 0, uses absolute rate comparison.

      For _rescued_data (inverted=true):
        Monitors the NON-null rate instead of the null rate.
    """
    t_start  = datetime.now()
    params   = parse_custom_parameters(validation_config.get("custom_parameters"))
    inverted = params.get("invert", False)

    print(f"    [NULL_SPIKE] Column: {column_name} {'(inverted)' if inverted else ''}")

    warning  = float(validation_config["warning_threshold"])
    critical = float(validation_config["critical_threshold"])

    try:
        df = spark.table(fqn)
        df, value_col = resolve_nested_column(df, column_name)

        if not value_col:
            print(f"      Column not found. Skipping.")
            return None

        if biz_date_col and biz_date_col in df.columns:
            df_date = df.filter(
                F.col(biz_date_col) == F.lit(str(business_date))
            )
        else:
            df_date = df

        total_count = df_date.count()

        if total_count == 0:
            print(f"      No records for {business_date}. Skipping.")
            return None

        null_count = df_date.filter(F.col(value_col).isNull()).count()
        null_rate  = null_count / total_count

        # For inverted columns (_rescued_data)
        # monitor non-null rate instead
        if inverted:
            observed_rate = (total_count - null_count) / total_count
            rate_label    = "non_null_rate"
        else:
            observed_rate = null_rate
            rate_label    = "null_rate"

        baseline_rate = baseline["mean_value"] if baseline else 0.0

        # ── Scoring Logic ──────────────────────────────────────────────────
        #
        # Case A: warning_threshold = 0
        #   Column should NEVER have nulls (e.g. transaction_id, event_id)
        #   Any null at all triggers WARNING
        #   Use absolute rate comparison
        #
        # Case B: warning_threshold > 0 and baseline_rate = 0
        #   Column normally has no nulls but we allow a small rate
        #   Compare observed rate directly against threshold
        #
        # Case C: warning_threshold > 0 and baseline_rate > 0
        #   Column has a natural null rate (e.g. customer_id at 35%)
        #   Compare how much observed rate EXCEEDS baseline
        #   Not the absolute rate

        observed_pct  = observed_rate * 100
        baseline_pct  = (baseline_rate or 0) * 100

        if warning == 0.0:
            # Case A: Zero-tolerance column
            # Any null is a warning, use critical threshold as absolute rate
            if observed_pct >= critical:
                status    = STATUS_CRITICAL
                deviation = observed_pct
            elif null_count > 0:
                status    = STATUS_WARNING
                deviation = observed_pct
            else:
                status    = STATUS_PASS
                deviation = 0.0

        elif baseline_pct == 0.0:
            # Case B: No historical nulls, compare absolute rate
            if observed_pct >= critical:
                status    = STATUS_CRITICAL
                deviation = observed_pct
            elif observed_pct >= warning:
                status    = STATUS_WARNING
                deviation = observed_pct
            else:
                status    = STATUS_PASS
                deviation = observed_pct

        else:
            # Case C: Natural null rate exists
            # Compare how much observed EXCEEDS baseline
            # deviation = observed_pct - baseline_pct
            # (positive = more nulls than normal = bad)
            deviation = observed_pct - baseline_pct

            if deviation >= critical:
                status = STATUS_CRITICAL
            elif deviation >= warning:
                status = STATUS_WARNING
            else:
                status = STATUS_PASS

        duration = (datetime.now() - t_start).total_seconds()

        detail = {
            "column_name"       : column_name,
            "total_records"     : total_count,
            "null_count"        : null_count,
            rate_label          : round(observed_pct, 4),
            "baseline_rate_pct" : round(baseline_pct, 4),
            "deviation_pct"     : round(deviation, 4),
            "inverted"          : inverted,
            "scoring_case"      : (
                "A_zero_tolerance"   if warning == 0.0
                else "B_no_baseline" if baseline_pct == 0.0
                else "C_natural_rate"
            )
        }

        print(f"      Total records   : {total_count}")
        print(f"      Null count      : {null_count}")
        print(f"      {rate_label}    : {observed_pct:.4f}%")
        print(f"      Baseline rate   : {baseline_pct:.4f}%")
        print(f"      Deviation       : {deviation:.4f}%")
        print(f"      Status          : {status}")

        return build_result(
            validation_run_id  = run_id,
            fqn                = fqn,
            business_date      = business_date,
            check_type         = "NULL_SPIKE",
            check_level        = "COLUMN",
            column_name        = column_name,
            check_status       = status,
            observed_value     = round(observed_pct, 6),
            baseline_value     = round(baseline_pct, 6),
            deviation_pct      = round(deviation, 4),
            warning_threshold  = warning,
            critical_threshold = critical,
            check_detail       = detail,
            duration_seconds   = duration
        )

    except Exception as e:
        print(f"      ERROR: {str(e)}")
        return build_result(
            validation_run_id  = run_id,
            fqn                = fqn,
            business_date      = business_date,
            check_type         = "NULL_SPIKE",
            check_level        = "COLUMN",
            column_name        = column_name,
            check_status       = STATUS_ERROR,
            observed_value     = None,
            baseline_value     = None,
            deviation_pct      = None,
            warning_threshold  = warning,
            critical_threshold = critical,
            check_detail       = {"error": str(e)},
            duration_seconds   = (datetime.now() - t_start).total_seconds()
        )

def check_numeric_range(
    fqn               : str,
    business_date     : date,
    biz_date_col      : str,
    column_name       : str,
    validation_config : dict,
    baseline          : dict,
    run_id            : str
) -> dict:
    """
    Check: NUMERIC_RANGE
    Counts records where column values fall outside
    the configured min and max bounds.
    """
    t_start = datetime.now()
    print(f"    [NUMERIC_RANGE] Column: {column_name}")

    warning  = float(validation_config["warning_threshold"])
    critical = float(validation_config["critical_threshold"])
    params   = parse_custom_parameters(validation_config.get("custom_parameters"))

    min_val = params.get("min_value")
    max_val = params.get("max_value")

    try:
        df = spark.table(fqn)
        df, value_col = resolve_nested_column(df, column_name)

        if not value_col:
            print(f"      Column not found. Skipping.")
            return None

        if biz_date_col and biz_date_col in df.columns:
            df_date = df.filter(
                F.col(biz_date_col) == F.lit(str(business_date))
            )
        else:
            df_date = df

        total_count = df_date.filter(
            F.col(value_col).isNotNull()
        ).count()

        if total_count == 0:
            return None

        # Build violation filter
        violation_condition = F.lit(False)
        if min_val is not None:
            violation_condition = violation_condition | (
                F.col(value_col) < F.lit(min_val)
            )
        if max_val is not None:
            violation_condition = violation_condition | (
                F.col(value_col) > F.lit(max_val)
            )

        violation_count = df_date.filter(
            F.col(value_col).isNotNull()
        ).filter(violation_condition).count()

        violation_rate = violation_count / total_count if total_count > 0 else 0.0
        violation_pct  = violation_rate * 100

        if violation_pct >= critical:
            status = STATUS_CRITICAL
        elif violation_pct >= warning:
            status = STATUS_WARNING
        else:
            status = STATUS_PASS

        duration = (datetime.now() - t_start).total_seconds()

        detail = {
            "column_name"     : column_name,
            "total_records"   : total_count,
            "violation_count" : violation_count,
            "violation_pct"   : round(violation_pct, 4),
            "configured_min"  : min_val,
            "configured_max"  : max_val
        }

        print(f"      Total records   : {total_count}")
        print(f"      Violations      : {violation_count} ({violation_pct:.4f}%)")
        print(f"      Range           : {min_val} to {max_val}")
        print(f"      Status          : {status}")

        return build_result(
            validation_run_id  = run_id,
            fqn                = fqn,
            business_date      = business_date,
            check_type         = "NUMERIC_RANGE",
            check_level        = "COLUMN",
            column_name        = column_name,
            check_status       = status,
            observed_value     = round(violation_pct, 6),
            baseline_value     = 0.0,
            deviation_pct      = round(violation_pct, 4),
            warning_threshold  = warning,
            critical_threshold = critical,
            check_detail       = detail,
            duration_seconds   = duration
        )

    except Exception as e:
        print(f"      ERROR: {str(e)}")
        return build_result(
            validation_run_id  = run_id,
            fqn                = fqn,
            business_date      = business_date,
            check_type         = "NUMERIC_RANGE",
            check_level        = "COLUMN",
            column_name        = column_name,
            check_status       = STATUS_ERROR,
            observed_value     = None,
            baseline_value     = None,
            deviation_pct      = None,
            warning_threshold  = warning,
            critical_threshold = critical,
            check_detail       = {"error": str(e)},
            duration_seconds   = (datetime.now() - t_start).total_seconds()
        )


def check_categorical_domain(
    fqn               : str,
    business_date     : date,
    biz_date_col      : str,
    column_name       : str,
    validation_config : dict,
    baseline          : dict,
    run_id            : str
) -> dict:
    """
    Check: CATEGORICAL_DOMAIN
    Detects values not in the configured valid value set.
    """
    t_start  = datetime.now()
    print(f"    [CATEGORICAL_DOMAIN] Column: {column_name}")

    warning  = float(validation_config["warning_threshold"])
    critical = float(validation_config["critical_threshold"])
    params   = parse_custom_parameters(validation_config.get("custom_parameters"))
    valid_values = params.get("valid_values", [])

    if not valid_values:
        print(f"      No valid_values configured. Skipping.")
        return None

    try:
        df = spark.table(fqn)
        df, value_col = resolve_nested_column(df, column_name)

        if not value_col:
            print(f"      Column not found. Skipping.")
            return None

        if biz_date_col and biz_date_col in df.columns:
            df_date = df.filter(
                F.col(biz_date_col) == F.lit(str(business_date))
            )
        else:
            df_date = df

        total_count = df_date.count()
        if total_count == 0:
            return None

        invalid_count = df_date.filter(
            ~F.col(value_col).isin(valid_values)
        ).count()

        invalid_pct = (invalid_count / total_count) * 100

        # Get the actual invalid values for the detail
        invalid_values_df = (
            df_date.select(F.col(value_col).alias("value_col"))
                   .filter(~F.col("value_col").isin(valid_values))
                   .groupBy("value_col")
                   .count()
                   .orderBy(F.col("count").desc())
                   .limit(10)
                   .collect()
        )
        invalid_value_map = {
            row["value_col"]: row["count"]
            for row in invalid_values_df
        }

        if invalid_pct >= critical:
            status = STATUS_CRITICAL
        elif invalid_pct >= warning:
            status = STATUS_WARNING
        else:
            status = STATUS_PASS

        duration = (datetime.now() - t_start).total_seconds()

        detail = {
            "column_name"        : column_name,
            "total_records"      : total_count,
            "invalid_count"      : invalid_count,
            "invalid_pct"        : round(invalid_pct, 4),
            "valid_values"       : valid_values,
            "invalid_values_found": invalid_value_map
        }

        print(f"      Total records   : {total_count}")
        print(f"      Invalid count   : {invalid_count} ({invalid_pct:.4f}%)")
        print(f"      Invalid values  : {invalid_value_map}")
        print(f"      Status          : {status}")

        return build_result(
            validation_run_id  = run_id,
            fqn                = fqn,
            business_date      = business_date,
            check_type         = "CATEGORICAL_DOMAIN",
            check_level        = "COLUMN",
            column_name        = column_name,
            check_status       = status,
            observed_value     = round(invalid_pct, 6),
            baseline_value     = 0.0,
            deviation_pct      = round(invalid_pct, 4),
            warning_threshold  = warning,
            critical_threshold = critical,
            check_detail       = detail,
            duration_seconds   = duration
        )

    except Exception as e:
        print(f"      ERROR: {str(e)}")
        return build_result(
            validation_run_id  = run_id,
            fqn                = fqn,
            business_date      = business_date,
            check_type         = "CATEGORICAL_DOMAIN",
            check_level        = "COLUMN",
            column_name        = column_name,
            check_status       = STATUS_ERROR,
            observed_value     = None,
            baseline_value     = None,
            deviation_pct      = None,
            warning_threshold  = warning,
            critical_threshold = critical,
            check_detail       = {"error": str(e)},
            duration_seconds   = (datetime.now() - t_start).total_seconds()
        )


def check_cardinality_drift(
    fqn               : str,
    business_date     : date,
    biz_date_col      : str,
    column_name       : str,
    validation_config : dict,
    baseline          : dict,
    run_id            : str
) -> dict:
    """
    Check: CARDINALITY_DRIFT
    Detects when the number of distinct values in a column
    changes significantly compared to the baseline.
    """
    t_start  = datetime.now()
    print(f"    [CARDINALITY_DRIFT] Column: {column_name}")

    warning  = float(validation_config["warning_threshold"])
    critical = float(validation_config["critical_threshold"])

    try:
        df = spark.table(fqn)
        df, value_col = resolve_nested_column(df, column_name)

        if not value_col:
            print(f"      Column not found. Skipping.")
            return None

        if biz_date_col and biz_date_col in df.columns:
            df_date = df.filter(
                F.col(biz_date_col) == F.lit(str(business_date))
            )
        else:
            df_date = df

        current_distinct = df_date.select(F.col(value_col).alias("value_col")).distinct().count()
        baseline_mean    = baseline["mean_value"] if baseline else None

        if baseline_mean and baseline_mean > 0:
            deviation_pct = abs(
                (current_distinct - baseline_mean) / baseline_mean
            ) * 100
        else:
            deviation_pct = 0.0

        if deviation_pct >= critical:
            status = STATUS_CRITICAL
        elif deviation_pct >= warning:
            status = STATUS_WARNING
        else:
            status = STATUS_PASS

        duration = (datetime.now() - t_start).total_seconds()

        detail = {
            "column_name"         : column_name,
            "current_distinct"    : current_distinct,
            "baseline_mean"       : baseline_mean,
            "baseline_min"        : baseline["min_value"] if baseline else None,
            "baseline_max"        : baseline["max_value"] if baseline else None,
            "deviation_pct"       : round(deviation_pct, 4),
            "direction"           : "INCREASE"
                                    if current_distinct > (baseline_mean or 0)
                                    else "DECREASE"
        }

        print(f"      Current distinct : {current_distinct}")
        print(f"      Baseline mean    : {baseline_mean}")
        print(f"      Deviation        : {deviation_pct:.2f}%")
        print(f"      Status           : {status}")

        return build_result(
            validation_run_id  = run_id,
            fqn                = fqn,
            business_date      = business_date,
            check_type         = "CARDINALITY_DRIFT",
            check_level        = "COLUMN",
            column_name        = column_name,
            check_status       = status,
            observed_value     = float(current_distinct),
            baseline_value     = baseline_mean,
            deviation_pct      = round(deviation_pct, 4),
            warning_threshold  = warning,
            critical_threshold = critical,
            check_detail       = detail,
            duration_seconds   = duration
        )

    except Exception as e:
        print(f"      ERROR: {str(e)}")
        return build_result(
            validation_run_id  = run_id,
            fqn                = fqn,
            business_date      = business_date,
            check_type         = "CARDINALITY_DRIFT",
            check_level        = "COLUMN",
            column_name        = column_name,
            check_status       = STATUS_ERROR,
            observed_value     = None,
            baseline_value     = None,
            deviation_pct      = None,
            warning_threshold  = warning,
            critical_threshold = critical,
            check_detail       = {"error": str(e)},
            duration_seconds   = (datetime.now() - t_start).total_seconds()
        )


def check_cross_column_consistency(
    fqn               : str,
    business_date     : date,
    biz_date_col      : str,
    column_name       : str,
    validation_config : dict,
    baseline          : dict,
    run_id            : str
) -> dict:
    """
    Check: CROSS_COLUMN_CONSISTENCY
    Evaluates a configured business rule across multiple columns.
    Counts records that violate the rule.

    Supported rules:
      raw_store_transactions:
        gross_amount - discount_amount + tax_amount = net_amount

      raw_inventory_movements:
        quantity_before + quantity_change = quantity_after
    """
    t_start = datetime.now()
    print(f"    [CROSS_COLUMN_CONSISTENCY] Column: {column_name}")

    warning   = float(validation_config["warning_threshold"])
    critical  = float(validation_config["critical_threshold"])
    params    = parse_custom_parameters(validation_config.get("custom_parameters"))
    rule      = params.get("rule", "")
    tolerance = float(params.get("tolerance", 0.01))

    try:
        df = spark.table(fqn)

        if biz_date_col and biz_date_col in df.columns:
            df_date = df.filter(
                F.col(biz_date_col) == F.lit(str(business_date))
            )
        else:
            df_date = df

        total_count = df_date.count()
        if total_count == 0:
            return None

        # Apply the consistency rule based on the rule string
        if "gross_amount - discount_amount + tax_amount = net_amount" in rule:
            df_check = df_date.filter(
                F.col("transaction_type").isin(["SALE", "EXCHANGE"])
            )
            check_total = df_check.count()

            if check_total == 0:
                return None

            violation_count = df_check.filter(
                F.abs(
                    F.col("gross_amount") -
                    F.coalesce(F.col("discount_amount"), F.lit(0.0)) +
                    F.coalesce(F.col("tax_amount"), F.lit(0.0)) -
                    F.col("net_amount")
                ) > tolerance
            ).count()

        elif "quantity_before + quantity_change = quantity_after" in rule:
            check_total = total_count
            violation_count = df_date.filter(
                F.abs(
                    F.col("quantity_before") +
                    F.col("quantity_change") -
                    F.col("quantity_after")
                ) > tolerance
            ).count()

        else:
            print(f"      Unknown rule: {rule}. Skipping.")
            return None

        violation_pct = (violation_count / check_total) * 100 if check_total > 0 else 0.0

        if violation_pct >= critical:
            status = STATUS_CRITICAL
        elif violation_pct >= warning:
            status = STATUS_WARNING
        else:
            status = STATUS_PASS

        duration = (datetime.now() - t_start).total_seconds()

        detail = {
            "rule"            : rule,
            "tolerance"       : tolerance,
            "total_checked"   : check_total,
            "violation_count" : violation_count,
            "violation_pct"   : round(violation_pct, 4)
        }

        print(f"      Records checked : {check_total}")
        print(f"      Violations      : {violation_count} ({violation_pct:.4f}%)")
        print(f"      Status          : {status}")

        return build_result(
            validation_run_id  = run_id,
            fqn                = fqn,
            business_date      = business_date,
            check_type         = "CROSS_COLUMN_CONSISTENCY",
            check_level        = "COLUMN",
            column_name        = column_name,
            check_status       = status,
            observed_value     = round(violation_pct, 6),
            baseline_value     = 0.0,
            deviation_pct      = round(violation_pct, 4),
            warning_threshold  = warning,
            critical_threshold = critical,
            check_detail       = detail,
            duration_seconds   = duration
        )

    except Exception as e:
        print(f"      ERROR: {str(e)}")
        return build_result(
            validation_run_id  = run_id,
            fqn                = fqn,
            business_date      = business_date,
            check_type         = "CROSS_COLUMN_CONSISTENCY",
            check_level        = "COLUMN",
            column_name        = column_name,
            check_status       = STATUS_ERROR,
            observed_value     = None,
            baseline_value     = None,
            deviation_pct      = None,
            warning_threshold  = warning,
            critical_threshold = critical,
            check_detail       = {"error": str(e)},
            duration_seconds   = (datetime.now() - t_start).total_seconds()
        )


def check_malformed_records(
    fqn               : str,
    business_date     : date,
    biz_date_col      : str,
    validation_config : dict,
    baseline          : dict,
    run_id            : str
) -> dict:
    """
    Check: MALFORMED_RECORDS
    For Autoloader tables, monitors the non-null rate of _rescued_data.
    A non-null _rescued_data means Autoloader could not parse that record.
    """
    t_start  = datetime.now()
    print(f"    [MALFORMED_RECORDS] Running...")

    warning  = float(validation_config["warning_threshold"])
    critical = float(validation_config["critical_threshold"])
    params   = parse_custom_parameters(validation_config.get("custom_parameters"))
    indicator_col = params.get("indicator_column", "_rescued_data")

    try:
        df = spark.table(fqn)

        if indicator_col not in df.columns:
            print(f"      Indicator column {indicator_col} not found. Skipping.")
            return None

        if biz_date_col and biz_date_col in df.columns:
            df_date = df.filter(
                F.col(biz_date_col) == F.lit(str(business_date))
            )
        else:
            df_date = df

        total_count    = df_date.count()
        malformed_count = df_date.filter(
            F.col(indicator_col).isNotNull()
        ).count()

        malformed_pct = (malformed_count / total_count) * 100 \
                        if total_count > 0 else 0.0

        if malformed_pct >= critical:
            status = STATUS_CRITICAL
        elif malformed_pct >= warning:
            status = STATUS_WARNING
        else:
            status = STATUS_PASS

        duration = (datetime.now() - t_start).total_seconds()

        detail = {
            "indicator_column" : indicator_col,
            "total_records"    : total_count,
            "malformed_count"  : malformed_count,
            "malformed_pct"    : round(malformed_pct, 4),
            "baseline_pct"     : 0.0
        }

        print(f"      Total records   : {total_count}")
        print(f"      Malformed       : {malformed_count} ({malformed_pct:.4f}%)")
        print(f"      Status          : {status}")

        return build_result(
            validation_run_id  = run_id,
            fqn                = fqn,
            business_date      = business_date,
            check_type         = "MALFORMED_RECORDS",
            check_level        = "TABLE",
            column_name        = None,
            check_status       = status,
            observed_value     = round(malformed_pct, 6),
            baseline_value     = 0.0,
            deviation_pct      = round(malformed_pct, 4),
            warning_threshold  = warning,
            critical_threshold = critical,
            check_detail       = detail,
            duration_seconds   = duration
        )

    except Exception as e:
        print(f"      ERROR: {str(e)}")
        return build_result(
            validation_run_id  = run_id,
            fqn                = fqn,
            business_date      = business_date,
            check_type         = "MALFORMED_RECORDS",
            check_level        = "TABLE",
            column_name        = None,
            check_status       = STATUS_ERROR,
            observed_value     = None,
            baseline_value     = None,
            deviation_pct      = None,
            warning_threshold  = warning,
            critical_threshold = critical,
            check_detail       = {"error": str(e)},
            duration_seconds   = (datetime.now() - t_start).total_seconds()
        )


print("All check functions loaded.")

All check functions loaded.


## Step 4: Write Results to Output Tables

In [0]:
def write_validation_results(results: list) -> bool:
    """
    Writes check results to uc1_validation_results_b.
    """
    if not results:
        return True

    column_order = [f.name for f in VALIDATION_RESULTS_SCHEMA.fields]
    rows = [tuple(r.get(col) for col in column_order) for r in results]

    df = spark.createDataFrame(rows, schema=VALIDATION_RESULTS_SCHEMA)

    df.write.mode("append").partitionBy("business_date").saveAsTable(
        VALIDATION_RESULTS
    )

    print(f"  Written {len(results)} result records to {VALIDATION_RESULTS}")
    return True


def write_validation_run(run_record: dict) -> bool:
    """
    Writes the run summary record to uc1_validation_run_b.
    """
    column_order = [f.name for f in VALIDATION_RUN_SCHEMA.fields]
    rows = [tuple(run_record.get(col) for col in column_order)]

    df = spark.createDataFrame(rows, schema=VALIDATION_RUN_SCHEMA)

    df.write.mode("append").partitionBy("business_date").saveAsTable(
        VALIDATION_RUN
    )

    print(f"  Written run record to {VALIDATION_RUN}")
    return True


def evaluate_confidence(results: list) -> dict:
    """
    Evaluates whether the AI confidence threshold is met.

    Criteria:
      1. At least 2 checks failed (WARNING or CRITICAL)
      2. At least 1 check is CRITICAL

    Returns a dict with the evaluation outcome.
    """
    failed_checks   = [
        r for r in results
        if r["check_status"] in (STATUS_WARNING, STATUS_CRITICAL)
    ]
    critical_checks = [
        r for r in results
        if r["check_status"] == STATUS_CRITICAL
    ]

    threshold_met = (
        len(failed_checks)   >= AI_MIN_FAILED_CHECKS and
        len(critical_checks) >= AI_MIN_CRITICAL_COUNT
    )

    return {
        "threshold_met"       : threshold_met,
        "failed_check_count"  : len(failed_checks),
        "critical_check_count": len(critical_checks),
        "failed_checks"       : [r["check_type"] for r in failed_checks],
        "skip_reason"         : None if threshold_met else (
            "INSUFFICIENT_FAILURES"
            if len(failed_checks) < AI_MIN_FAILED_CHECKS
            else "NO_CRITICAL_CHECKS"
        )
    }


def determine_overall_status(results: list) -> str:
    """
    Determines the overall run status from individual check results.
    Returns the worst status across all checks.
    """
    if not results:
        return STATUS_PASS

    worst = STATUS_PASS
    for r in results:
        if SEVERITY_RANK.get(r["check_status"], 0) > SEVERITY_RANK.get(worst, 0):
            worst = r["check_status"]
    return worst


print("Write functions loaded.")

Write functions loaded.


## Step 5: Run Validation for All Active Tables

In [0]:
active_tables      = load_active_tables(TABLE_FILTER)
all_run_records    = []
all_result_records = []

print(f"\nValidating {len(active_tables)} table(s) for {BUSINESS_DATE}\n")

for table_row in active_tables:

    fqn          = table_row["fully_qualified_name"]
    biz_date_col = table_row.get("business_date_column") or ""
    table_results = []

    # ── Generate a unique run ID per table ────────────────────────────────
    # Each table gets its own UUID so AI enrichment queries
    # return checks from only one table at a time.
    TABLE_RUN_ID = str(uuid.uuid4())

    print(f"\n{'='*60}")
    print(f"Table          : {fqn}")
    print(f"Run ID         : {TABLE_RUN_ID}")
    print(f"Business date  : {BUSINESS_DATE}")
    print(f"{'='*60}")

    # ── Load configuration ─────────────────────────────────────────────────
    table_validations = load_table_validations(fqn)

    col_val_rows    = load_column_validations(fqn)
    col_validations = {}
    for cv in col_val_rows:
        col_name = cv["column_name"]
        if col_name not in col_validations:
            col_validations[col_name] = []
        col_validations[col_name].append(cv)

    print(f"Table validations  : {len(table_validations)}")
    print(f"Column validations : {sum(len(v) for v in col_validations.values())}")

    # ── Table-Level Checks ─────────────────────────────────────────────────
    print(f"\n  TABLE-LEVEL CHECKS")

    for tv in table_validations:
        val_name = tv["validation_name"]

        if val_name == "VOLUME_ANOMALY":
            baseline = load_baseline(fqn, "VOLUME_ANOMALY", "__table__")
            result   = check_volume_anomaly(
                fqn, BUSINESS_DATE, biz_date_col,
                tv, baseline, TABLE_RUN_ID
            )
            if result:
                table_results.append(result)

        elif val_name == "SCHEMA_CONFORMANCE":
            baseline = load_baseline(fqn, "SCHEMA_CONFORMANCE", "__table__")
            result   = check_schema_conformance(
                fqn, BUSINESS_DATE,
                tv, baseline, TABLE_RUN_ID
            )
            if result:
                table_results.append(result)

        elif val_name == "MALFORMED_RECORDS":
            baseline = load_baseline(fqn, "MALFORMED_RECORDS", "__table__")
            result   = check_malformed_records(
                fqn, BUSINESS_DATE, biz_date_col,
                tv, baseline, TABLE_RUN_ID
            )
            if result:
                table_results.append(result)

        elif val_name in ("PARTITION_FRESHNESS", "DUPLICATE_INGESTION"):
            print(f"    [{val_name}] Skipped in this version")

    # ── Column-Level Checks ────────────────────────────────────────────────
    print(f"\n  COLUMN-LEVEL CHECKS")

    for col_name, validations in col_validations.items():
        for cv in validations:
            val_name = cv["validation_name"]

            if val_name == "NULL_SPIKE":
                baseline = load_baseline(fqn, "NULL_SPIKE", col_name)
                result   = check_null_spike(
                    fqn, BUSINESS_DATE, biz_date_col,
                    col_name, cv, baseline, TABLE_RUN_ID
                )
                if result:
                    table_results.append(result)

            elif val_name == "NUMERIC_RANGE":
                baseline = load_baseline(fqn, "NUMERIC_RANGE", col_name)
                result   = check_numeric_range(
                    fqn, BUSINESS_DATE, biz_date_col,
                    col_name, cv, baseline, TABLE_RUN_ID
                )
                if result:
                    table_results.append(result)

            elif val_name == "CATEGORICAL_DOMAIN":
                baseline = load_baseline(
                    fqn, "CATEGORICAL_DOMAIN", col_name
                )
                result   = check_categorical_domain(
                    fqn, BUSINESS_DATE, biz_date_col,
                    col_name, cv, baseline, TABLE_RUN_ID
                )
                if result:
                    table_results.append(result)

            elif val_name == "CARDINALITY_DRIFT":
                baseline = load_baseline(
                    fqn, "CARDINALITY_DRIFT", col_name
                )
                result   = check_cardinality_drift(
                    fqn, BUSINESS_DATE, biz_date_col,
                    col_name, cv, baseline, TABLE_RUN_ID
                )
                if result:
                    table_results.append(result)

            elif val_name == "CROSS_COLUMN_CONSISTENCY":
                baseline = load_baseline(
                    fqn, "CROSS_COLUMN_CONSISTENCY", col_name
                )
                result   = check_cross_column_consistency(
                    fqn, BUSINESS_DATE, biz_date_col,
                    col_name, cv, baseline, TABLE_RUN_ID
                )
                if result:
                    table_results.append(result)

    # ── Evaluate Results ───────────────────────────────────────────────────
    overall_status  = determine_overall_status(table_results)
    confidence      = evaluate_confidence(table_results)
    ai_status       = (
        "PENDING"
        if confidence["threshold_met"]
        else "NOT_REQUIRED"
    )

    checks_passed   = sum(
        1 for r in table_results
        if r["check_status"] == STATUS_PASS
    )
    checks_warned   = sum(
        1 for r in table_results
        if r["check_status"] == STATUS_WARNING
    )
    checks_critical = sum(
        1 for r in table_results
        if r["check_status"] == STATUS_CRITICAL
    )
    checks_errored  = sum(
        1 for r in table_results
        if r["check_status"] == STATUS_ERROR
    )

    run_end_time = datetime.now()
    duration     = (run_end_time - RUN_START_TIME).total_seconds()

    # ── Build Run Record ───────────────────────────────────────────────────
    run_record = {
        "validation_run_id"           : TABLE_RUN_ID,
        "fully_qualified_name"        : fqn,
        "business_date"               : BUSINESS_DATE,
        "ingestion_timestamp"         : datetime.now(),
        "triggered_by"                : "MANUAL",
        "pipeline_run_id"             : None,
        "validation_start_timestamp"  : RUN_START_TIME,
        "validation_end_timestamp"    : run_end_time,
        "validation_duration_seconds" : duration,
        "overall_status"              : overall_status,
        "total_checks_run"            : len(table_results),
        "total_checks_passed"         : checks_passed,
        "total_checks_warned"         : checks_warned,
        "total_checks_critical"       : checks_critical,
        "total_checks_errored"        : checks_errored,
        "ai_enrichment_status"        : ai_status,
        "ai_enrichment_timestamp"     : None,
        "created_timestamp"           : datetime.now(),
        "created_by"                  : "06_validation_agent"
    }

    all_run_records.append(run_record)
    all_result_records.extend(table_results)

    # ── Print Table Summary ────────────────────────────────────────────────
    print(f"\n  {'─'*50}")
    print(f"  RESULTS SUMMARY: {fqn}")
    print(f"  {'─'*50}")
    print(f"  Run ID            : {TABLE_RUN_ID}")
    print(f"  Overall status    : {overall_status}")
    print(f"  Total checks run  : {len(table_results)}")
    print(f"  Passed            : {checks_passed}")
    print(f"  Warning           : {checks_warned}")
    print(f"  Critical          : {checks_critical}")
    print(f"  Errored           : {checks_errored}")
    print(f"  AI enrichment     : {ai_status}")
    if confidence["threshold_met"]:
        print(f"  AI triggered by   : {confidence['failed_checks']}")
    else:
        print(f"  AI skip reason    : {confidence['skip_reason']}")
    print(f"  {'─'*50}")

print(f"\nAll tables processed.")
print(f"Total result records : {len(all_result_records)}")
print(f"Total run records    : {len(all_run_records)}")


Validating 1 table(s) for 2026-10-14


Table          : comms_media_prd.exposure_data_asset.bronze_barrows_campaign
Run ID         : ea02b5d9-db2a-45d6-9967-2721a1124b01
Business date  : 2026-10-14
Table validations  : 4
Column validations : 29

  TABLE-LEVEL CHECKS
    [VOLUME_ANOMALY] Running...
      Current rows    : 0
      Baseline mean   : 4.357142857142857
      Deviation       : -100.00%
      Status          : CRITICAL
    [MALFORMED_RECORDS] Running...
      Total records   : 0
      Malformed       : 0 (0.0000%)
      Status          : PASS

  COLUMN-LEVEL CHECKS
    [NULL_SPIKE] Column: id 
      No records for 2026-10-14. Skipping.
    [CARDINALITY_DRIFT] Column: id
      Current distinct : 0
      Baseline mean    : 1.0
      Deviation        : 100.00%
      Status           : CRITICAL
    [NULL_SPIKE] Column: created 
      No records for 2026-10-14. Skipping.
    [NULL_SPIKE] Column: updated 
      No records for 2026-10-14. Skipping.
    [CROSS_COLUMN_CONSISTENCY] Co

## Step 6: Write Results to Delta Tables

In [0]:
print("Writing validation results...")

for run_record in all_run_records:
    write_validation_run(run_record)

write_validation_results(all_result_records)

print("\nWrite complete.")

Writing validation results...
  Written run record to learning_dev.s819534.uc1_validation_run_b
  Written 4 result records to learning_dev.s819534.uc1_validation_results_b

Write complete.


## Step 7: View Results for This Run

In [0]:
print("=" * 60)
print(f"VALIDATION RESULTS — {BUSINESS_DATE}")
print("=" * 60)

# Run summary
print("\n1. Validation Run Summary:\n")
spark.sql(f"""
    SELECT
        validation_run_id,
        fully_qualified_name,
        business_date,
        overall_status,
        total_checks_run,
        total_checks_passed,
        total_checks_warned,
        total_checks_critical,
        total_checks_errored,
        ai_enrichment_status,
        ROUND(validation_duration_seconds, 2) AS duration_seconds
    FROM {VALIDATION_RUN}
    WHERE validation_run_id = '{VALIDATION_RUN_ID}'
    ORDER BY fully_qualified_name
""").show(truncate=False)

# Detailed check results
print("\n2. Individual Check Results:\n")
spark.sql(f"""
    SELECT
        fully_qualified_name,
        check_type,
        check_level,
        column_name,
        check_status,
        ROUND(observed_value, 4)    AS observed_value,
        ROUND(baseline_value, 4)    AS baseline_value,
        ROUND(deviation_percentage, 2) AS deviation_pct,
        threshold_warning,
        threshold_critical,
        ROUND(check_duration_seconds, 3) AS duration_sec
    FROM {VALIDATION_RESULTS}
    WHERE validation_run_id = '{VALIDATION_RUN_ID}'
    ORDER BY
        fully_qualified_name,
        CASE check_status
            WHEN 'CRITICAL' THEN 1
            WHEN 'WARNING'  THEN 2
            WHEN 'ERROR'    THEN 3
            WHEN 'PASS'     THEN 4
            ELSE 5
        END,
        check_type
""").show(100, truncate=False)

# Failed checks detail
print("\n3. Failed Checks Detail (WARNING and CRITICAL only):\n")
spark.sql(f"""
    SELECT
        fully_qualified_name,
        check_type,
        column_name,
        check_status,
        check_detail
    FROM {VALIDATION_RESULTS}
    WHERE validation_run_id = '{VALIDATION_RUN_ID}'
      AND check_status IN ('WARNING', 'CRITICAL')
    ORDER BY
        CASE check_status
            WHEN 'CRITICAL' THEN 1
            WHEN 'WARNING'  THEN 2
        END,
        check_type
""").show(100, truncate=False)

VALIDATION RESULTS — 2026-10-14

1. Validation Run Summary:

+-----------------+--------------------+-------------+--------------+----------------+-------------------+-------------------+---------------------+--------------------+--------------------+----------------+
|validation_run_id|fully_qualified_name|business_date|overall_status|total_checks_run|total_checks_passed|total_checks_warned|total_checks_critical|total_checks_errored|ai_enrichment_status|duration_seconds|
+-----------------+--------------------+-------------+--------------+----------------+-------------------+-------------------+---------------------+--------------------+--------------------+----------------+
+-----------------+--------------------+-------------+--------------+----------------+-------------------+-------------------+---------------------+--------------------+--------------------+----------------+


2. Individual Check Results:

+--------------------+----------+-----------+-----------+------------+-----

## Final Summary

In [0]:
print("\n" + "=" * 60)
print("AGENT 1 VALIDATION COMPLETE")
print("=" * 60)
print(f"Business date      : {BUSINESS_DATE}")
print(f"Validation run ID  : {VALIDATION_RUN_ID}")
print(f"Tables validated   : {len(all_run_records)}")
print(f"Total checks run   : {len(all_result_records)}")
print(f"Completed at       : {datetime.now()}")
print()

for run in all_run_records:
    print(f"  {run['fully_qualified_name']}")
    print(f"    Status   : {run['overall_status']}")
    print(f"    Passed   : {run['total_checks_passed']}")
    print(f"    Warning  : {run['total_checks_warned']}")
    print(f"    Critical : {run['total_checks_critical']}")
    print(f"    AI       : {run['ai_enrichment_status']}")
    print()

print("=" * 60)


AGENT 1 VALIDATION COMPLETE
Business date      : 2026-10-14
Validation run ID  : 0b22e45c-65b8-4afc-80a0-baae6fb8410b
Tables validated   : 1
Total checks run   : 4
Completed at       : 2026-11-11 16:31:58.200066

  comms_media_prd.exposure_data_asset.bronze_barrows_campaign
    Status   : CRITICAL
    Passed   : 1
    Critical : 3
    AI       : PENDING

